# PIPELINE DATA CLEANING — HOUSING DATASET

Nama  : Hadhist Rizqi Fauzhi

NIM   : 250401020093

Kelas : IF405

Import Library

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests

Muat Dataset

In [ ]:
df = pd.read_csv('housing_dirty.csv')

Eksplorasi Awal

In [ ]:
print(df.info())
print(df.describe())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    87.050000  3.450000e+02    2.000000   1991.250000
50%     65.50000

Hapus Duplikat

In [ ]:
df.drop_duplicates(inplace=True)
print("Sisa data:", df.shape)

Sisa data: (130, 7)


Normalisasi String

In [ ]:
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

Imputasi Missing Values

In [ ]:
# numerik → median
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

# kategorik → modus
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

Tangani Outlier (IQR Fence)

In [ ]:
for col in ['harga_juta', 'luas_m2']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

Validasi Akhir

In [ ]:
assert df.isnull().sum().sum() == 0, "Masih ada missing!"
assert df.duplicated().sum() == 0, "Masih ada duplikat!"
print("Total missing:", df.isnull().sum().sum())
print("Total duplikat:", df.duplicated().sum())

Total missing: 0
Total duplikat: 0


Ekspor Data

In [ ]:
df.to_csv('housing_clean.csv', index=False)
print("Dataset bersih tersimpan!")

Dataset bersih tersimpan!


Akses API (JSONPlaceholder)

In [ ]:
import requests
from pandas import json_normalize

url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url, timeout=10)

if response.status_code == 200:
    data = response.json()
    df_api = json_normalize(data)
    print(df_api.head())
else:
    print("Error:", response.status_code)

   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    010-692-6593 x09125  anastasia.net      Victor Plains     Suite 879   
2         1-463-123-4447    ramiro.info  Douglas Extension     Suite 847   
3      493-170-9623 x156       kale.biz        Hoeger Mall      Apt. 692   
4          (254)954-1289   demarco.info       Skiles Walks     Suite 351   

    address.city address.zipcode address.geo.lat address.geo.lng  \
0    Gwenborough      92998-3874        -37.3159         81.1496   
1    Wisokyburgh

##Kesimpulan

Program ini menunjukkan alur data cleaning yang cukup lengkap pada dataset perumahan sebelum data digunakan untuk analisis lebih lanjut. Proses dimulai dengan memuat dataset dan melakukan eksplorasi awal untuk melihat struktur data, statistik dasar, serta jumlah nilai yang hilang.

Selanjutnya dilakukan pembersihan data melalui beberapa tahap: menghapus baris duplikat, menormalkan penulisan teks pada kolom kategori, mengisi nilai yang hilang dengan metode yang sesuai (median untuk data numerik dan modus untuk data kategorik), serta menangani outlier menggunakan metode IQR Fence agar nilai yang terlalu ekstrem tidak mendistorsi analisis.

Pada tahap validasi, program memastikan bahwa tidak ada lagi missing value maupun data duplikat yang tersisa. Setelah data dinyatakan bersih, hasilnya disimpan ke file baru sehingga dataset awal tetap terjaga dan dataset yang telah diproses dapat langsung digunakan untuk tahap analisis berikutnya.

Bagian terakhir menunjukkan cara mengambil data dari API eksternal menggunakan JSONPlaceholder dan mengubah respons JSON menjadi bentuk tabel menggunakan Pandas. Ini memperlihatkan bahwa sumber data tidak hanya berasal dari file lokal, tetapi juga dapat diperoleh dari layanan web.

Secara keseluruhan, kode ini menggambarkan praktik penting dalam Data Science, yaitu memastikan kualitas data terlebih dahulu sebelum melakukan analisis atau pemodelan. Data yang sudah dibersihkan dan divalidasi akan lebih konsisten, lebih mudah dianalisis, dan menghasilkan insight yang lebih dapat dipercaya.